<a href="https://colab.research.google.com/github/JMan003/RAG-LLM/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Kaggle: set up working directories
import os
os.makedirs('/kaggle/working/lawracle/chroma_db', exist_ok=True)
print('Directories ready.')
print('PDFs expected at: /kaggle/input/datasets/johnjoset/dataset/')


In [ ]:
# Install required packages
!pip install -q transformers bitsandbytes accelerate sentence_transformers llama-index langchain pyngrok flask torch chromadb llama-index-vector-stores-chroma pymupdf

: 

In [ ]:
!pip install -q llama-index-llms-huggingface

In [ ]:
!pip install -q langchain_community

In [ ]:
!pip install -q llama-index-embeddings-langchain

In [ ]:
import os
import torch
import logging
import sys
from tqdm.notebook import tqdm
from flask import Flask, request, jsonify
from pyngrok import ngrok
from llama_index.core import VectorStoreIndex, Document, Settings, StorageContext
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core.prompts.prompts import SimpleInputPrompt
from langchain_community.embeddings import HuggingFaceEmbeddings
import fitz  # PyMuPDF
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

# Set up logging
logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger("pypdf").setLevel(logging.ERROR)

# Create Flask app
app = Flask(__name__)

# Create data directory if it doesn't exist
!mkdir -p data

# Extract text from PDF using PyMuPDF
def extract_text_from_pdf(pdf_path):
    try:
        text = ""
        with fitz.open(pdf_path) as doc:
            for page in doc:
                text += page.get_text()
        return text
    except Exception as e:
        print(f"Error extracting text from {pdf_path}: {str(e)}")
        return ""

# Function to load and chunk documents with progress bar
def load_and_process_documents():
    print("Loading documents...")
    documents = []
    data_dir = '/kaggle/input/datasets/johnjoset/dataset'

    # Count files first
    files = [f for f in os.listdir(data_dir) if os.path.isfile(os.path.join(data_dir, f))]

    for filename in tqdm(files, desc="Processing files"):
        file_path = os.path.join(data_dir, filename)
        try:
            if filename.lower().endswith('.pdf'):
                text = extract_text_from_pdf(file_path)
                if not text.strip():
                    print(f"Warning: No text extracted from {filename}")
                    continue
            elif filename.lower().endswith(('.txt', '.md')):
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    text = f.read()
            else:
                continue

            documents.append(Document(text=text, metadata={"source": filename}))
            print(f"Successfully loaded {filename} ({len(text)} characters)")
        except Exception as e:
            print(f"Error processing {filename}: {str(e)}")

    return documents

# Create index from documents with aggressive chunking and progress reporting
def create_vector_index(documents, embed_model, chroma_collection):
    print(f"Creating vector index from {len(documents)} documents...")

    # Use a more aggressive text splitter to create smaller chunks
    text_splitter = SentenceSplitter(
        chunk_size=512,  # Larger chunks capture full legal provisions
        chunk_overlap=50,
        paragraph_separator="\n\n",
        separator=" "
    )

    print("Splitting documents into chunks...")
    nodes = []
    for doc in tqdm(documents, desc="Chunking documents"):
        doc_nodes = text_splitter.get_nodes_from_documents([doc])
        nodes.extend(doc_nodes)

    print(f"Created {len(nodes)} text chunks")

    # Create ChromaDB-backed vector store and persist to disk
    print("Building vector index with ChromaDB...")
    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex(nodes, storage_context=storage_context)

    return index

# Function to truncate text to a safe length
def truncate_text(text, max_words=200):
    """Truncate text to approximately max_words"""
    words = text.split()
    if len(words) <= max_words:
        return text
    return ' '.join(words[:max_words]) + "..."

# Format response in LEAP structure with length limits
def format_leap_response(query, raw_response):
    """Format the response according to the L.E.A.P. structure with length limits"""

    # Check if response already has LEAP structure and is reasonably sized
    if "Legal Issue" in raw_response and "Action Steps" in raw_response:
        words = raw_response.split()
        if len(words) <= 400:  # If already in LEAP format and not too long
            return raw_response

    # Simple assessment of query type to customize response
    query_lower = query.lower()

    # Identify potential legal issues based on keywords
    if any(word in query_lower for word in ["accident", "crash", "collision"]):
        issue = "a motor vehicle accident and potential liability under road safety laws"
    elif any(word in query_lower for word in ["property", "land", "tenant", "owner"]):
        issue = "a property dispute or real estate matter"
    elif any(word in query_lower for word in ["marriage", "divorce", "custody"]):
        issue = "a family law matter"
    elif any(word in query_lower for word in ["job", "salary", "employer", "fired"]):
        issue = "an employment law issue"
    else:
        issue = "a potential legal concern requiring professional advice"

    # Truncate raw response to a reasonable length
    truncated_response = truncate_text(str(raw_response), 150)

    # Format the response with controlled length for each section
    formatted_response = f"""
**Legal Issue**:
This query concerns {issue}.

**Explanation of Law**:
{truncated_response}

**Action Steps**:
1. Consult with a qualified legal professional specializing in this area of law
2. Gather all relevant documentation and evidence related to your case
3. Consider filing appropriate paperwork with relevant authorities

**Practical Guidance**:
Maintain thorough documentation of all events, communications, and expenses related to this matter.
"""
    return formatted_response

# Main function to set up the system
def setup_rag_system():
    print("Setting up RAG system...")

    # Define the Lawracle system prompt - shortened for token efficiency
    system_prompt = """
You are Lawracle, an AI legal assistant specializing in Indian law.

STRICT RULES:
- Answer ONLY using information explicitly stated in the provided context documents.
- Do NOT invent, assume, or add legal provisions, punishments, or section numbers not present in the context.
- If the context lacks information, state: 'The available documents do not contain specific information on this point.'
- Always cite exact section numbers and act names as found in the context.

Use the L.E.A.P. structure:
1. Legal Issue: 1-2 sentences identifying the core legal problem.
2. Explanation of Law: 2-3 sentences citing only what the context states.
3. Action Steps: 2-3 practical steps.
4. Practical Guidance: 1-2 tips.

Keep your TOTAL response strictly under 150 words.
"""

    # Define query wrapper prompt
    query_wrapper_prompt = SimpleInputPrompt(
    "<s>[INST] " + system_prompt + "\n\n{query_str} [/INST]"
)

    # Load Mistral-7B with 4-bit quantization to fit in T4 VRAM
    llm_model_name = "/kaggle/input/models/mistral-ai/mistral/pytorch/7b-instruct-v0.1-hf/1"
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
    model = AutoModelForCausalLM.from_pretrained(
        llm_model_name,
        quantization_config=bnb_config,
        device_map={"": 0},  # Force all layers to GPU 0
        torch_dtype=torch.float16,
    )

    # Use a smaller, faster embedding model
    embed_model = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2",  # Much smaller embedding model
        model_kwargs={"device": "cpu"}
    )

    # Wire model into LlamaIndex properly
    llm = HuggingFaceLLM(
        model=model,
        tokenizer=tokenizer,
        context_window=3900,
        max_new_tokens=512,
        #system_prompt=system_prompt,  # Embedded in query_wrapper_prompt instead
        query_wrapper_prompt=query_wrapper_prompt,
        generate_kwargs={"temperature": 0.2, "do_sample": True},
    )

    # Configure settings
    Settings.llm = llm
    Settings.embed_model = embed_model
    Settings.chunk_size = 512
    Settings.chunk_overlap = 50

    # Initialize ChromaDB persistent client
    chroma_client = chromadb.PersistentClient(path="/kaggle/working/lawracle/chroma_db")
    chroma_collection = chroma_client.get_or_create_collection("lawracle_docs")

    # Load from cache if embeddings already exist (skips re-embedding)
    if chroma_collection.count() > 0:
        print(f"Loading existing ChromaDB collection ({chroma_collection.count()} embeddings cached)...")
        vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
        index = VectorStoreIndex.from_vector_store(vector_store)
    else:
        # First run: load, embed, and persist documents
        documents = load_and_process_documents()

        if not documents:
            print("No documents were loaded. Creating a sample document...")
            sample_text = "Sample legal text about Motor Vehicles Act Section 134."
            documents = [Document(text=sample_text, metadata={"source": "sample.txt"})]

        # Create index and persist to ChromaDB
        index = create_vector_index(documents, embed_model, chroma_collection)

    # Create query engine with response length limits
    query_engine = index.as_query_engine(
        response_mode="compact",
        similarity_top_k=4,  # More chunks = broader legal context
    )

    return query_engine

# API route for queries with error handling for length issues
@app.route('/query', methods=['POST'])
def process_query():
    data = request.json
    if not data or 'query' not in data:
        return jsonify({"error": "No query provided"}), 400

    query = data['query']
    print(f"Processing query: {query}")

    try:
        # Get raw response from query engine - handle possible exceptions
        try:
            raw_response = query_engine.query(query)
        except ValueError as e:
            if "exceed the model's predefined maximum length" in str(e):
                print("Warning: Response exceeded length limit. Using truncated context.")
                raw_response = "The retrieved information was too lengthy. Please ask a more specific question."
            else:
                raise e

        # Format in LEAP structure with length control
        formatted_response = format_leap_response(query, str(raw_response))

        return jsonify({"response": formatted_response})
    except Exception as e:
        print(f"Error: {str(e)}")
        # Fallback response if there's an error
        fallback_response = format_leap_response(query, "Unable to retrieve specific information for your query.")
        return jsonify({"response": fallback_response})

In [ ]:
# ── Initialise the RAG system ──────────────────────────────────────────────
print('Initializing RAG system...')
query_engine = setup_rag_system()
print('\n✅ RAG system ready!')

In [ ]:
# ── Test the RAG system with sample queries ────────────────────────────────
test_queries = [
    "What is the punishment for theft under the Bharatiya Nyaya Sanhita?",
    "What are the grounds for divorce under the Hindu Marriage Act?",
    "What are my rights if a cheque I received bounces?",
    "What constitutes an offence under the Motor Vehicles Act?",
]

for query in test_queries:
    print(f"\n{'='*65}")
    print(f'QUERY: {query}')
    print('='*65)
    response = query_engine.query(query)
    print(format_leap_response(query, str(response)))

# ── Or test your own query ────────────────────────────────────────────────
# custom_query = "your question here"
# response = query_engine.query(custom_query)
# print(format_leap_response(custom_query, str(response)))

In [ ]:
# ── Launch the public API via Flask + ngrok ────────────────────────────────
# Run this cell only when you want to expose the API publicly.
# Requires an ngrok auth token from https://dashboard.ngrok.com

import threading

ngrok_auth_token = input('Enter your ngrok auth token (press Enter to skip): ')

if ngrok_auth_token:
    ngrok.set_auth_token(ngrok_auth_token)
    public_url = ngrok.connect(5000)
    print(f'\n✅ API is publicly available at: {public_url}/query')
    print('Send POST requests with JSON body: {"query": "your legal question"}')

# Run Flask in a background thread so Colab doesn't block
flask_thread = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=5000, use_reloader=False)
)
flask_thread.daemon = True
flask_thread.start()
print('Flask server running on port 5000')